# CC-Tempo v2 — Kaggle runner

**How to use (every time):**
1. Set `STAGE` in the next cell (see the table; follow `TODO_PLAN.md` in order). Several stages can run in one version, joined by `+` (e.g. `'A+B+C'`); they run in that order.
2. Click **Save Version → Save & Run All (Commit)**. Kaggle runs it in the background (up to 12 h; you can close the browser).
3. When the version finishes, open it → **Output** → download `cct_outputs.zip` and send it to Claude.
4. For the next stage: **+ Add Input → Your Work → this notebook (latest version)** so finished data/runs are restored, change `STAGE`, Save & Run All again.

Settings (right panel): Accelerator **GPU T4 x2**, Internet **On**. Data stages (A, F1, G1) can use Accelerator *None* to save GPU quota.

| STAGE | what | GPU? | time (approx.) |
|---|---|---|---|
| `A`  | Weinreb download + LIANA + dataset | no | 1–1.5 h |
| `B`  | synthetic ground-truth benchmark | optional | 1–2 h |
| `C`  | timing calibration (50 epochs, 2 models) | yes | 10–20 min |
| `D`  | Weinreb tier-1 training (20 runs, both GPUs) | yes | 1–3 versions (resumes automatically) |
| `E`  | Weinreb evaluation + baselines + interpretation | yes | 2–4 h |
| `W2` | Weinreb tier-2 (LIANA-method robustness, controls, held-out Day 4) | yes | 1–2 versions |
| `F1` | cytokine data prep (+ clone evidence) | no | 30–45 min |
| `F2` | cytokine training (joint + leave-one-medium-out) + eval + dose-response | yes | 1–2 versions |
| `G1` | neural-tube download + inspect + annotate | no | 30 min |
| `G2` | neural-tube LIANA + training + analysis (only after Claude has checked `inspect.txt`) | yes | 1–2 versions |


In [ ]:
STAGE = 'A+B+C'      # <-- change before each Save & Run All; join stages with '+'
EXTRA_SET = []       # optional config overrides, e.g. ['train.epochs=1000']

In [ ]:
# setup: latest code from GitHub, install, restore earlier outputs
import os, glob, subprocess, sys, shutil
REPO = 'https://github.com/Ahnaf-Akib-Tanim/cc-tempo_v2.git'
CODE = '/kaggle/working/cc-tempo_v2'
ROOT = '/kaggle/working'
if os.path.exists(CODE):
    subprocess.run(['git', '-C', CODE, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', REPO, CODE], check=True)
os.chdir(CODE)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt'], check=True)
os.environ.update(CCT_ROOT=ROOT, POT_BACKEND_DISABLE_TENSORFLOW='1', POT_BACKEND_DISABLE_JAX='1', PYTHONUNBUFFERED='1')
import time
SESSION_HOURS = 10.75   # Kaggle kills a version at 12 h; training stops cleanly before this
os.environ['CCT_DEADLINE'] = str(time.time() + SESSION_HOURS * 3600)

# restore work/ from earlier versions of this notebook (Add Input -> Your Work) or an uploaded zip
for src in glob.glob('/kaggle/input/*/work'):
    print('restoring', src)
    shutil.copytree(src, f'{ROOT}/work', dirs_exist_ok=True)
for z in glob.glob('/kaggle/input/*/cct_outputs*.zip'):
    print('unzipping', z)
    subprocess.run(['unzip', '-o', '-q', z, '-d', ROOT])
print(subprocess.run('find /kaggle/working/work -maxdepth 3 2>/dev/null | head -60', shell=True, capture_output=True, text=True).stdout)

In [ ]:
def run(cmd):
    '''Run one pipeline command and stream its output; on failure print and continue.'''
    if EXTRA_SET and '--set' not in cmd:
        cmd += ' --set ' + ' '.join(EXTRA_SET)
    print('\n$', cmd, flush=True)
    rc = subprocess.run(cmd, shell=True).returncode
    if rc != 0:
        print(f'!!! FAILED ({rc}): {cmd}', flush=True)
    return rc


def grid(dataset, tier):
    '''Run one tier of configs/experiments.yaml, one shard per GPU in parallel, and wait.'''
    import torch
    n = max(1, torch.cuda.device_count())
    procs = []
    for k in range(n):
        dev = f'cuda:{k}' if torch.cuda.is_available() else 'cpu'
        c = f'python scripts/06_run_grid.py --dataset {dataset} --tier {tier} --shard {k}/{n} --device {dev} --root {ROOT}'
        if EXTRA_SET:
            c += ' --set ' + ' '.join(EXTRA_SET)
        print('\n$', c, flush=True)
        procs.append(subprocess.Popen(c + f' > {ROOT}/grid_{dataset}_{tier}_{k}.log 2>&1', shell=True))
    for p in procs:
        p.wait()
    print(subprocess.run(f'tail -n 15 {ROOT}/grid_{dataset}_{tier}_*.log', shell=True, capture_output=True, text=True).stdout)


run('python scripts/00_check_env.py')

In [ ]:
R = f'--root {ROOT}'


def run_stage(st):
    if st == 'A':      # cheap to repeat: step 1 skips itself when its outputs exist
        run(f'python scripts/01_prepare_weinreb.py --config weinreb.yaml {R}')
        run(f'python scripts/02_liana_ccc.py --config weinreb.yaml {R}')
        run(f'python scripts/03_build_dataset.py --config weinreb.yaml {R}')
        shutil.rmtree(f'{ROOT}/raw/weinreb', ignore_errors=True)   # 2 GB download no longer needed
    elif st == 'B':
        run(f'python scripts/04_synthetic_benchmark.py {R}')
    elif st == 'C':
        for preset in ('v2_meanfield', 'prescient'):
            run(f'python scripts/05_train.py --config weinreb.yaml --preset {preset} --seed 99 --tag timing {R} '
                '--set train.epochs=50 train.eval_every=25')
        run(f'tail -n 2 {ROOT}/work/weinreb/runs/*timing*/train_log.csv')
    elif st == 'D':
        grid('weinreb', 'tier1')
    elif st == 'E':
        run(f'python scripts/07_evaluate.py --config weinreb.yaml {R}')
        run(f'python scripts/08_baselines.py --config weinreb.yaml {R} --methods wot cfm')
        run(f'python scripts/09_interpret.py --config weinreb.yaml {R} --run v2_meanfield_s0')
    elif st == 'W2':
        grid('weinreb', 'tier2')
        run(f'python scripts/07_evaluate.py --config weinreb.yaml {R}')
    elif st == 'F1':
        run(f'python scripts/10_cytokine_experiments.py --config cytokine.yaml --evidence {R}')
        run(f'python scripts/01_prepare_weinreb.py --config cytokine.yaml {R}')
        run(f'python scripts/02_liana_ccc.py --config cytokine.yaml {R}')
        run(f'python scripts/03_build_dataset.py --config cytokine.yaml {R}')
        shutil.rmtree(f'{ROOT}/raw/weinreb_cytokine', ignore_errors=True)
    elif st == 'F2':
        grid('cytokine', 'tier1')
        grid('cytokine', 'tier2')
        run(f'python scripts/07_evaluate.py --config cytokine.yaml {R}')
        run(f'python scripts/10_cytokine_experiments.py --config cytokine.yaml --dose-response --run v2_meanfield_s0 {R}')
    elif st == 'G1':
        run(f'python scripts/11_neural_tube.py --prepare {R}')
        shutil.rmtree(f'{ROOT}/raw/neural_tube', ignore_errors=True)
        run(f'cat {ROOT}/work/neural_tube/inspect.txt')
    elif st == 'G2':
        run(f'python scripts/02_liana_ccc.py --config neural_tube.yaml {R}')
        run(f'python scripts/03_build_dataset.py --config neural_tube.yaml {R}')
        grid('neural_tube', 'tier1')
        run(f'python scripts/07_evaluate.py --config neural_tube.yaml {R}')
        run(f'python scripts/11_neural_tube.py --analyze --run v2_meanfield_s0 {R}')
    else:
        raise ValueError(f'unknown stage {st}')


for st in [x.strip() for x in STAGE.split('+') if x.strip()]:
    print('=' * 20, 'STAGE', st, '=' * 20, flush=True)
    run_stage(st)

In [ ]:
# always last: small zip of results/metrics/logs/models to download and send to Claude
for f in glob.glob(f'{ROOT}/grid_*.log'):
    shutil.copy(f, f'{ROOT}/work/')               # logs travel with work/ into the next version
run(f'python scripts/pack_outputs.py --root {ROOT} --with-models')
os.chdir(ROOT)
shutil.rmtree(CODE, ignore_errors=True)            # code lives on GitHub; keep the version output small
print(subprocess.run(f'ls -lh {ROOT}; du -sh {ROOT}/work', shell=True, capture_output=True, text=True).stdout)